# Hafta 5: Ev Fiyat Tahmini - California Housing

Bu defterde sklearn'ün California Housing veri setini kullanarak:
- Keşifsel veri analizi (EDA)
- Korelasyon tabanlı özellik seçimi
- Lineer Regresyon vs Polinom Regresyon karşılaştırması
- Artık (residual) analizi
- Çapraz doğrulama (Cross Validation)

konularını uygulayacağız.

## 1. Gerekli Kütüphaneler

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Kütüphaneler başarıyla yüklendi!")

## 2. Veri Setini Yükleme

### Özellik Seçimi ve Mühendisliği

Model için kullanılacak özellikleri belirliyoruz. Doğru özellik seçimi model performansını doğrudan etkiler.

In [ ]:
# California Housing veri seti
housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # Hedef: Medyan ev değeri (100.000$ cinsinden)

print(f"Veri seti boyutu: {df.shape}")
print(f"Özellik sayısı: {len(housing.feature_names)}")
print(f"\nÖzellik açıklamaları:")
print("-" * 60)
aciklamalar = {
    'MedInc': 'Medyan gelir (10.000$ cinsinden)',
    'HouseAge': 'Medyan ev yaşı',
    'AveRooms': 'Ortalama oda sayısı',
    'AveBedrms': 'Ortalama yatak odası sayısı',
    'Population': 'Nüfus',
    'AveOccup': 'Ortalama doluluk oranı',
    'Latitude': 'Enlem',
    'Longitude': 'Boylam',
    'MedHouseVal': 'Medyan ev değeri (100.000$)'
}
for col, desc in aciklamalar.items():
    print(f"  {col:15s}: {desc}")

print(f"\nİlk 5 satır:")
df.head()

## 3. Keşifsel Veri Analizi (EDA)

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
# Temel istatistikler
df.describe().round(2)

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Özellik dağılımları
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

renkler = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6',
           '#1abc9c', '#e67e22', '#34495e', '#c0392b']

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color=renkler[i], edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Tüm Özelliklerin Dağılımları', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Korelasyon ısı haritası
plt.figure(figsize=(12, 10))
corr = df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))  # Üst üçgeni maskeleme
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Korelasyon Katsayısı'})
plt.title('Korelasyon Isı Haritası', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Hedef değişken ile korelasyon sıralama
hedef_korelasyon = corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)

print("MedHouseVal ile Korelasyonlar (Sıralı):")
print("=" * 45)
for ozellik, deger in hedef_korelasyon.items():
    bar = '█' * int(abs(deger) * 20)
    isaret = '+' if deger > 0 else '-'
    print(f"  {ozellik:15s} {isaret}{abs(deger):.3f}  {bar}")
print("=" * 45)

## 4. Özellik Seçimi

Korelasyon analizine göre en etkili özellikleri seçeceğiz.

In [ ]:
# Mutlak korelasyonu 0.1'den büyük olan özellikleri seçelim
esik = 0.1
secilen_ozellikler = hedef_korelasyon[abs(hedef_korelasyon) > esik].index.tolist()

print(f"Seçilen özellikler (|korelasyon| > {esik}):")
for ozellik in secilen_ozellikler:
    print(f"  - {ozellik} (r = {hedef_korelasyon[ozellik]:.3f})")

print(f"\nToplam {len(secilen_ozellikler)} özellik seçildi (orijinal: {len(housing.feature_names)})")

### Saçılım Grafiği

İki değişken arasındaki ilişkiyi saçılım grafiği ile inceliyoruz. Noktaların oluşturduğu desen, doğrusal veya doğrusal olmayan ilişkiyi gösterir.

In [ ]:
# En güçlü özelliklerle scatter plotlar
en_guclu = hedef_korelasyon.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ozellik in enumerate(en_guclu):
    axes[i].scatter(df[ozellik], df['MedHouseVal'], alpha=0.1, s=5, color=renkler[i])
    axes[i].set_xlabel(ozellik)
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{ozellik} vs Ev Değeri (r={hedef_korelasyon[ozellik]:.3f})')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('En Güçlü Korelasyona Sahip Özellikler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Eğitim/Test Ayrımı

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Özellikler ve hedef
X = df[secilen_ozellikler]
y = df['MedHouseVal']

# %80 eğitim, %20 test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Eğitim seti: {X_train.shape[0]} örnek, {X_train.shape[1]} özellik")
print(f"Test seti  : {X_test.shape[0]} örnek, {X_test.shape[1]} özellik")

## 6. Lineer Regresyon vs Polinom Regresyon Karşılaştırması

In [ ]:
# Model 1: Lineer Regresyon
model_lineer = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
model_lineer.fit(X_train, y_train)
y_pred_lineer = model_lineer.predict(X_test)

# Model 2: Polinom Regresyon (Derece 2)
model_poly2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
model_poly2.fit(X_train, y_train)
y_pred_poly2 = model_poly2.predict(X_test)

# Model 3: Polinom Regresyon (Derece 3)
model_poly3 = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
model_poly3.fit(X_train, y_train)
y_pred_poly3 = model_poly3.predict(X_test)

print("Modeller başarıyla eğitildi!")

### Metrik karşılaştırma tablosu

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Metrik karşılaştırma tablosu
def metrikleri_hesapla(y_gercek, y_tahmin):
    return {
        'MAE': mean_absolute_error(y_gercek, y_tahmin),
        'RMSE': np.sqrt(mean_squared_error(y_gercek, y_tahmin)),
        'R²': r2_score(y_gercek, y_tahmin)
    }

sonuclar = {
    'Lineer Regresyon': metrikleri_hesapla(y_test, y_pred_lineer),
    'Polinom (Derece 2)': metrikleri_hesapla(y_test, y_pred_poly2),
    'Polinom (Derece 3)': metrikleri_hesapla(y_test, y_pred_poly3),
}

sonuc_df = pd.DataFrame(sonuclar).T
sonuc_df = sonuc_df.round(4)

print("\n" + "=" * 60)
print("  MODEL KARŞILAŞTIRMA TABLOSU")
print("=" * 60)
print(sonuc_df.to_string())
print("=" * 60)

en_iyi = sonuc_df['R²'].idxmax()
print(f"\nEn iyi model: {en_iyi} (R² = {sonuc_df.loc[en_iyi, 'R²']:.4f})")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Metrik karşılaştırma görselleştirmesi
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_isimleri = list(sonuclar.keys())
renkler_bar = ['#3498db', '#e74c3c', '#2ecc71']

for i, metrik in enumerate(['MAE', 'RMSE', 'R²']):
    degerler = [sonuclar[m][metrik] for m in model_isimleri]
    bars = axes[i].bar(model_isimleri, degerler, color=renkler_bar, edgecolor='gray')
    axes[i].set_title(metrik, fontweight='bold', fontsize=14)
    axes[i].grid(True, alpha=0.3, axis='y')
    axes[i].tick_params(axis='x', rotation=15)
    
    # Değerleri barların üstüne yazma
    for bar, deger in zip(bars, degerler):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                     f'{deger:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Karşılaştırması', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Artık (Residual) Analizi

Artık analizi, modelin hatalarını incelememizi sağlar. İyi bir modelde artıklar:
- Rastgele dağılmalı (örüntü olmamalı)
- Ortalama 0'a yakın olmalı
- Normal dağılıma yakın olmalı

In [ ]:
# Lineer Regresyon için artık analizi
artiklar = y_test - y_pred_lineer

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Artık vs Tahmin
axes[0, 0].scatter(y_pred_lineer, artiklar, alpha=0.3, s=10, color='steelblue')
axes[0, 0].axhline(y=0, color='red', linewidth=2, linestyle='--')
axes[0, 0].set_xlabel('Tahmin Edilen Değer')
axes[0, 0].set_ylabel('Artık (Residual)')
axes[0, 0].set_title('Artık vs Tahmin')
axes[0, 0].grid(True, alpha=0.3)

# 2. Artık dağılımı (histogram)
axes[0, 1].hist(artiklar, bins=50, color='coral', edgecolor='white', alpha=0.8, density=True)
x_range = np.linspace(artiklar.min(), artiklar.max(), 100)
from scipy import stats
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, artiklar.mean(), artiklar.std()),
                'r-', linewidth=2, label='Normal Dağılım')
axes[0, 1].set_xlabel('Artık Değeri')
axes[0, 1].set_ylabel('Yoğunluk')
axes[0, 1].set_title('Artık Dağılımı')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot
stats.probplot(artiklar, plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normal Dağılım Kontrolü)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Gerçek vs Tahmin
axes[1, 1].scatter(y_test, y_pred_lineer, alpha=0.3, s=10, color='mediumseagreen')
min_val = min(y_test.min(), y_pred_lineer.min())
max_val = max(y_test.max(), y_pred_lineer.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
axes[1, 1].set_xlabel('Gerçek Değer')
axes[1, 1].set_ylabel('Tahmin Edilen Değer')
axes[1, 1].set_title('Gerçek vs Tahmin')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Lineer Regresyon - Artık Analizi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Artık istatistikleri:")
print(f"  Ortalama : {artiklar.mean():.4f}")
print(f"  Std Sapma: {artiklar.std():.4f}")
print(f"  Min      : {artiklar.min():.4f}")
print(f"  Max      : {artiklar.max():.4f}")

## 8. Çapraz Doğrulama (Cross Validation)

Çapraz doğrulama, modelin farklı veri alt kümelerinde nasıl performans gösterdiğini ölçer. Bu sayede modelin genelleme yeteneği hakkında daha güvenilir bilgi elde ederiz.

In [ ]:
# Tüm veri üzerinde çapraz doğrulama (5-Fold)
print("5-Katlı Çapraz Doğrulama Sonuçları")
print("=" * 55)

modeller = {
    'Lineer Regresyon': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    'Polinom (Derece 2)': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
}

cv_sonuclar = []

for isim, model in modeller.items():
    # R² skoru ile çapraz doğrulama
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')
    # Negatif RMSE (sklearn convention)
    cv_rmse = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    
    cv_sonuclar.append({
        'Model': isim,
        'R² (Ort.)': cv_r2.mean(),
        'R² (Std)': cv_r2.std(),
        'RMSE (Ort.)': -cv_rmse.mean(),
        'RMSE (Std)': cv_rmse.std(),
    })
    
    print(f"\n{isim}:")
    print(f"  R²   : {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")
    print(f"  Her katlama: {[f'{x:.4f}' for x in cv_r2]}")

print("\n" + "=" * 55)

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Çapraz doğrulama sonuçlarını görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_isimleri_cv = [s['Model'] for s in cv_sonuclar]
r2_ort = [s['R² (Ort.)'] for s in cv_sonuclar]
r2_std = [s['R² (Std)'] for s in cv_sonuclar]
rmse_ort = [s['RMSE (Ort.)'] for s in cv_sonuclar]
rmse_std = [s['RMSE (Std)'] for s in cv_sonuclar]

axes[0].bar(model_isimleri_cv, r2_ort, yerr=r2_std, capsize=5,
            color=['#3498db', '#e74c3c'], edgecolor='gray', alpha=0.8)
axes[0].set_title('R² Skoru (5-Fold CV)', fontweight='bold')
axes[0].set_ylabel('R²')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(model_isimleri_cv, rmse_ort, yerr=rmse_std, capsize=5,
            color=['#3498db', '#e74c3c'], edgecolor='gray', alpha=0.8)
axes[1].set_title('RMSE (5-Fold CV)', fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Çapraz Doğrulama Karşılaştırması', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Özet

Bu defterde öğrendiklerimiz:

1. **California Housing** veri setini yükledik ve inceledik
2. **Korelasyon ısı haritası** ile özellikler arası ilişkileri analiz ettik
3. **Özellik seçimi** ile en etkili değişkenleri belirledik
4. **Lineer ve Polinom Regresyon** modellerini karşılaştırdık
5. **Artık analizi** ile model hatalarını detaylı inceledik
6. **Çapraz doğrulama** ile modelin genelleme yeteneğini ölçtük

### Önemli Bulgular
- **MedInc** (medyan gelir), ev fiyatını en çok etkileyen özellik
- Polinom regresyon lineerden daha iyi performans gösterebilir, ancak aşırı öğrenme riski vardır
- Çapraz doğrulama, tek train/test split'ten daha güvenilir sonuçlar verir